<a href="https://colab.research.google.com/github/Girawale-Sanket/Gen-AI-LLM-Assignment-/blob/main/English_to_Multilingual_Translator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 2) English to Multilingual Translator
import time
import ipywidgets as widgets
from IPython.display import display, HTML

try:
    import translators as ts
except ModuleNotFoundError:
    print("Installing missing dependency 'translators'...")
    !pip install -q translators
    import translators as ts

LANGS = {
    'Hindi': 'hi',        'Spanish': 'es',     'French': 'fr',
    'German': 'de',       'Arabic': 'ar',      'Chinese': 'zh',
    'Japanese': 'ja',     'Russian': 'ru',     'Portuguese': 'pt',
    'Tamil': 'ta',        'Telugu': 'te',      'Bengali': 'bn',
    'Korean': 'ko',       'Italian': 'it',     'Urdu': 'ur',
}

ENGINES = ['bing', 'google', 'alibaba', 'youdao']
_cache = {}

title = widgets.HTML(
    "<h2 style='font-family:Arial;color:#4A90E2;margin:0'>"
    " English ->Multilingual Translator</h2>"
    "<p style='color:#777;font-family:Arial;margin:2px 0 10px'>"
    "Type English, pick a language, hit Translate.</p>"
)

inp = widgets.Textarea(
    placeholder='Type English text here...',
    layout=widgets.Layout(width='520px', height='110px')
)
lang = widgets.Dropdown(
    options=list(LANGS.keys()), value='Hindi', description='To:',
    layout=widgets.Layout(width='220px')
)
btn = widgets.Button(
    description='Translate', button_style='primary', icon='language',
    layout=widgets.Layout(width='140px')
)
clear = widgets.Button(
    description='Clear', button_style='warning', icon='eraser',
    layout=widgets.Layout(width='110px')
)
out = widgets.Output()

def safe_translate(text, target_code):
    key = (text, target_code)
    if key in _cache:
        return _cache[key]

    errors = []
    for engine in ENGINES:
        try:
            result = ts.translate_text(
                text,
                translator=engine,
                from_language='en',
                to_language=target_code
            )
            if result and result.strip() and result.strip().lower() != text.strip().lower():
                _cache[key] = (result, engine)
                return _cache[key]
        except Exception as e:
            errors.append(f"{engine}: {e}")
            time.sleep(0.3)

    raise RuntimeError(" | ".join(errors))

def translate(_):
    out.clear_output()
    with out:
        text = inp.value.strip()
        if not text:
            display(HTML("<i style='color:#c0392b'> Please enter some text.</i>"))
            return
        display(HTML("<span style='color:#555'>Translating…</span>"))
        try:
            result, engine = safe_translate(text, LANGS[lang.value])
            out.clear_output()
            with out:
                display(HTML(
                    f"<div style='background:#E8F5E9;border-left:5px solid #43A047;"
                    f"padding:12px 14px;border-radius:8px;font-size:17px;"
                    f"font-family:Arial;color:#1B5E20'>{result}"
                    f"<div style='font-size:11px;color:#558b2f;margin-top:6px'>"
                    f"via {engine.upper()}</div></div>"
                ))
        except Exception as e:
            out.clear_output()
            with out:
                display(HTML(
                    "<div style='background:#FDECEA;border-left:5px solid #E53935;"
                    "padding:12px 14px;border-radius:8px;color:#b71c1c;"
                    "font-family:Arial'>"
                    f" All engines failed.<br><small>{e}</small><br>"
                    "<small>Wait a few seconds and try again.</small></div>"
                ))

def reset(_):
    inp.value = ''
    out.clear_output()

btn.on_click(translate)
clear.on_click(reset)

display(widgets.VBox([
    title,
    inp,
    widgets.HBox([lang, btn, clear]),
    out
], layout=widgets.Layout(
    padding='15px',
    border='1px solid #ddd',
    border_radius='10px'
)))